# S12 — Train From Scratch (50 epochs, No Pretraining)\nSame as S11 but without ImageNet weights.

## Setup

In [ ]:
import sys, json, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Configuration

In [ ]:
EPOCHS = 50
BATCH_SIZE = 8
LR = 1e-3
SEED = 42
OUTPUT_DIR = Path('../models/s11_finetune')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output: {OUTPUT_DIR}\nEpochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR}')

## Data Preparation

In [ ]:
from src.config import DEVICE, print_device_info
from src.utils import set_seed
from src.data_loader import DatasetConfig, DataPathManager, VolumeWiseSplitter, create_2d_dataloaders
from src.preprocessing import PreprocessingTransform, AugmentedPreprocessingTransform, CLAHEProcessor
from src.models import create_model, count_params
from src.trainer import Trainer

set_seed(SEED); print_device_info()
path_manager = DataPathManager()
volume_index = path_manager.build_index()
splitter = VolumeWiseSplitter()
splits = splitter.load_splits(DatasetConfig.SPLITS_DIR)
print(f'Splits: train={len(splits["train"])}, val={len(splits["val"])}, test={len(splits["test"])}')

clahe = CLAHEProcessor(clip=2.0, grid=(8, 8))
transform_train = AugmentedPreprocessingTransform(target_size=(256, 256), hu_low=-100, hu_high=400, clahe=clahe)
transform_val = PreprocessingTransform(target_size=(256, 256), hu_low=-100, hu_high=400)
train_loader, val_loader, test_loader = create_2d_dataloaders(
    volume_index, splits['train'], splits['val'], splits['test'],
    batch_size=BATCH_SIZE, transform_train=transform_train, transform_val=transform_val)
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

## Model & Trainer Setup

In [ ]:
model = create_model('mobilenetv2_unet', in_channels=1, out_channels=1, pretrained=True)
model = model.to(DEVICE)
print(f'Parameters: {count_params(model):,}')

from src.config import PHASE4_RESEARCH_CONFIG
train_config = dict(PHASE4_RESEARCH_CONFIG)
train_config['use_focal'] = True
trainer = Trainer(model=model, train_loader=train_loader, val_loader=val_loader,
    config=train_config, learning_rate=LR, num_epochs=EPOCHS,
    mixed_precision=True, output_dir=str(OUTPUT_DIR))
print('Trainer ready.')

## Train (50 epochs — ~4 hrs)\nRun overnight. Cells below can be re-run after training completes.

In [ ]:
print('Starting 50-epoch fine-tune...')
trainer.train(warmup_epochs=5)
print(f'Best val dice: {trainer.best_val_dice:.4f}')

## Save Results

In [ ]:
with open(OUTPUT_DIR / 'history.json', 'w') as f:
    json.dump(trainer.history, f, indent=2)
test_metrics = trainer.evaluate(test_loader)
print(f'Test Dice: {test_metrics["dice"]:.4f}, IoU: {test_metrics["iou"]:.4f}')
with open(OUTPUT_DIR / 'test_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)

## Training Curve

In [ ]:
import matplotlib.pyplot as plt
hist = trainer.history
plt.figure(figsize=(10, 4))
plt.subplot(1,2,1); plt.plot(hist['train_dice'], label='Train'); plt.plot(hist['val_dice'], label='Val'); plt.legend(); plt.title('Dice')
plt.subplot(1,2,2); plt.plot(hist['train_loss'], label='Train'); plt.plot(hist['val_loss'], label='Val'); plt.legend(); plt.title('Loss')
plt.savefig(OUTPUT_DIR / 'training_curve.png', dpi=150)
plt.show()
print(f'Best val_dice: {max(hist["val_dice"]):.4f}')